- See here Large langauge models (LLMs) are powerful but they two key limitations:
    - Finite context --> they cant ingest entire corpora at once.
    - static knoweldge --> Their training data is frozen at poin in time.

- Retrieval addresses these problems by fetching relevant external knowledge at query time. This is the foundation of Retrieval-Augmented Generation (RAG): enhancing an LLM’s answers with context-specific information.

#### Building a knowledge base
- A knoweldge base is a repository of documents or structured data used during retrieval
- If you need a custom knowledge base, you can use LangChain’s document loaders and vector stores to build one from your own data.

#### From retrieval to RAG
- Retrieval allows LLMs to access relevant context at runtime. But most real-world applications go one step further: they integrate retrieval with generation to produce grounded, context-aware answers.
- This is the core idea behind Retrieval-Augmented Generation (RAG). The retrieval pipeline becomes a foundation for a broader system that combines search with generation.

#### Retriveval Pipeline
- A typical retrieval workflow looks like this:

![alt text](image-1.png)
- Each component is modular: you can swap loaders, splitters, embeddings, or vector stores without rewriting the app’s logic.



- First understand the problem.
- suppose we have vector database with 50000 chunks:
```
Chunk 1 : Apache Spark Overview
Chunk 2 : Kafka Streams
Chunk 3 : Airflow DAGs
...
Chunk 50000 : HR Leave Policy
```

- here if user asks a query ```what is apache spark```
- here the question is should we send 50 k chunks to model .. no right obiviuously 
- becuase : too expensive, context window limitations, most chunks irrelvant. 
- we need a mechanism that fetches only the most relevant chunks , this mechanism is called retriver 


#### What is Retriever?
- A Retriever is a component that retrieves relevant documents from a knowledge source based on a query.
- flow is like the below 
```
User Question
      ↓
Retriever
      ↓
Top Relevant Chunks
      ↓
LLM
      ↓
Answer
```


Creating a Retriver

In [ ]:
vector_store = Pinecone.from_document(
    chunks,
    embeddings
)

# convert to retriver
retriver = vector_store.as_retriver()

# now 
docs = retriver.invoke(
    "What is Apache Spark?"
)

# it will returns the below 
[
 Document(...),
 Document(...)
]

# the documents are sent to the llm for giving the response.

A retriever is any object that takes a string query and returns a list of Document objects. It's the standard interface LangChain uses so chains and agents don't care how the search works underneath — vector store, keyword search, web search, database — it all looks the same.


In [ ]:
# every retriever has one method 
docs = retriever.invoke("what is langchain")
# returns the list of documentd list[Document]

The key distinction from vector store

In [ ]:
# Vector store — you call specific methods
vectorstore.similarity_search(query, k=4)
vectorstore.max_marginal_relevance_search(query, k=4)

# Retriever — one clean interface, works in any chain
retriever.invoke(query)   # that's it

1 — Vector Store Retriever (foundation)


In [10]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name = os.getenv("huggingface_model_name"))
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# basic top k similarity
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":4}
)

# MMR diverse results
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.6})

# Score threshold — reject low-confidence results
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.75, "k": 6}
)

# Score threshold — reject low-confidence results
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.75, "k": 6})

# Use it
docs = retriever.invoke("What is LangChain?")
for doc in docs:
    print(doc.metadata)
    print(doc.page_content[:150])
    print("---")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2392.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_core\vectorstores\base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='acc27ee2-8c5d-4a92-a155-7ca0aca2e900', metadata={'title': 'Product Manual'}, page_content='Product Manual'), -0.23820410830675032), (Document(id='82776ad6-5399-4bd3-a7f4-16954a06bfb5', metadata={'section': 'Installation', 'title': 'Product Manual'}, page_content='Installation'), -0.3436691788599249), (Document(id='3f9a17aa-e2fd-4968-9ba8-58a0f90db886', metadata={'section': 'Configuration', 'title': 'Product Manual'}

2 — MultiQueryRetriever
- The probelm it solves : A single query embedding can miss document phrased differently. if you ask "what is langchain" but a chunk says "lanchain provides chain for llms" the cosine similarity is might be low even though it is the right answer.

- the solution: Generate multiple rephrased versions of the query automatically and take the union of all results.


In [14]:
from re import search
import logging
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq
import logging

# see the generated quries in terminal
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

llm = ChatGroq(model = os.getenv("groq_model_name"))
retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(search_kwargs = {"k":3}),
    llm =llm
)
docs = retriever.invoke("how does langchain Handle Memoery")
print(f"Total unique docs recieved",{len(docs)})


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['Here are three different versions of the user question:', '1. ', 'What information does LangChain provide regarding its handling of memory allocation and management?', "This question is rephrased to focus on specific details related to memory handling, which may help retrieve documents that discuss the technical aspects of LangChain's memory management.", '2. ', 'How does LangChain optimize memory usage and mitigate potential memory-related issues in its workflow?', "This rephrased question highlights the optimization and mitigation aspects of memory handling, which might retrieve documents that discuss LangChain's strategies for efficient memory usage.", '3. ', 'Can you explain the approach LangChain takes to manage memory consumption and trade-offs in its architecture?', "This version of the question focuses on the design decisions and trade-offs involved in LangChain's memory management approach, which may help retri

Total unique docs recieved {3}


```
INFO: langchain_classic.retrievers.multi_query: Generatedqueries: [
  'Here are three different versions of the user question:',
  '1. ',
  'What information does LangChain provide regarding its handling of memory allocation and management?',
  "This question is rephrased to focus on specific details related to memory handling, which may help retrieve documents that discuss the technical aspects of LangChain's memory management.",
  '2. ',
  'How does LangChain optimize memory usage and mitigate potential memory-related issues in its workflow?',
  "This rephrased question highlights the optimization and mitigation aspects of memory handling, which might retrieve documents that discuss LangChain's strategies for efficient memory usage.",
  '3. ',
  'Can you explain the approach LangChain takes to manage memory consumption and trade-offs in its architecture?',
  "This version of the question focuses on the design decisions and trade-offs involved in LangChain's memory management approach, which may help retrieve documents that discuss the underlying architectural considerations and design principles."
]Totaluniquedocsrecieved{
  3
}
```

3 — ContextualCompressionRetriever

- The problem it solves: retrieved chunks often 1000 characters long but only two sentences are actually relevant to the question. here we are burning llm context window on noise.
- The solution : Pass each retrieved chunk through compressor that extracts only the relevant part before sending it the llm.

In [16]:
from langchain_classic.retrievers  import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_groq import ChatGroq

llm = ChatGroq(model = os.getenv("groq_model_name"))
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 5})
)
docs = compression_retriever.invoke("What was the Q4 2024 revenue?")

for doc in docs:
    print(doc.page_content)   # only the precise relevant sentences
    print("---")

In [18]:
# Using LLMChainFilter instead — faster, cheaper (filters whole chunks in/out rather than extracting sentences):
from langchain_classic.retrievers.document_compressors import LLMChainFilter

_filter = LLMChainFilter.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 6})
)
# Returns whole chunks, but only the relevant ones (others dropped entirely)

In [19]:
# Using EmbeddingsFilter — no LLM call, pure embedding similarity (fastest):
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter

embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.76
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 6})
)
# Drops any retrieved chunk whose embedding is below 0.76 similarity to query
# No LLM call — just cosine filtering. Very fast.

5 — EnsembleRetriever
- The Problem it Solves: vector search understands meaning but misses exact keywords. BM25 key word search matches exact terms but misses semantic paraphrases. Neither alone is optimal.

- The solution: Run both, merge using reciprocal rank Fusion(RRF). a result that ranks high in both lists gets boosted.

In [6]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv

# 1. Initialize embeddings and the vector store
load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name=os.getenv("huggingface_model_name"))
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# 2. Your documents
docs = [
    Document(page_content="LangChain is a framework for LLM applications.", metadata={"source": "intro"}),
    Document(page_content="FAISS is a library for efficient similarity search.", metadata={"source": "faiss"}),
    Document(page_content="RAG combines retrieval with language model generation.", metadata={"source": "rag"}),
    Document(page_content="Chroma is an open-source embedding database.", metadata={"source": "chroma"}),
    Document(page_content="Vector stores index embeddings for fast nearest-neighbour search.", metadata={"source": "vectors"}),
]

# 3. Retriever 1: BM25 keyword search (no embeddings needed)
bm25_retriever = BM25Retriever.from_documents(docs, k=3)

# 4. Retriever 2: vector similarity search
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 5. Combine with weights
ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]    # 40% BM25, 60% vector
)

results = ensemble.invoke("What is FAISS used for?")

for doc in results:
    print(doc.page_content)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3102.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Product Manual
Configuration
Installation
FAISS is a library for efficient similarity search.
Chroma is an open-source embedding database.
LangChain is a framework for LLM applications.


6 — WebResearchRetriever (live web search)
- when the knoweldge base doesnot have the answer, search the web at query time.

In [ ]:
from langchain_community.retrievers.web_research import WebResearchRetriever
from langchain_community.utilities import GoogleSearchAPIWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Temp vector store to cache web results
temp_vectorstore = Chroma(
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./web_cache"
)

search = GoogleSearchAPIWrapper()   # needs GOOGLE_API_KEY + GOOGLE_CSE_ID

retriever = WebResearchRetriever.from_llm(
    vectorstore=temp_vectorstore,
    llm=ChatOpenAI(model="gpt-4o-mini"),
    search=search,
    num_search_results=3
)

docs = retriever.invoke("Latest LangChain release notes 2025")
# Searches the web, fetches pages, splits and embeds them,
# stores in temp vectorstore, returns relevant chunks

7 — ParentDocumentRetriever (advanced)
- The problem it solves : Small chunks embed better (more precise meaning), but large chunks give the LLM more context. we cant have both unles we use parent documents.
- The solution: store small child chunks for embedding/search, but when a child chunk matches, return the full parent document instead.

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# Child splitter: small chunks for precise embedding
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

# Parent splitter: larger chunks returned to the LLM
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)

# Two stores: vector store for child chunks, doc store for parents
vectorstore = Chroma(
    collection_name="child_chunks",
    embedding_function=OpenAIEmbeddings()
)
docstore = InMemoryStore()   # stores parent documents by ID

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

# Ingest
loader = PyPDFLoader("docs/manual.pdf")
docs = loader.load()
retriever.add_documents(docs)

# At query time:
# 1. Small child chunks are searched (precise matching)
# 2. Their parent documents are returned (full context for LLM)
results = retriever.invoke("What are the installation steps?")
print(f"Result size: {len(results[0].page_content)} chars")
# Returns ~1000 char parent, not 200 char child

8 — TimeWeightedRetriever (recency-aware)
- Boosts recently accessed or recently added documents. Useful for chat history, news, or anything where freshness matters.


In [ ]:
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
import datetime

vectorstore = Chroma(embedding_function=OpenAIEmbeddings())

retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,       # 0=no decay, 1=full decay after 1 day
    k=4
)

# Add documents — last_accessed_at tracked automatically
docs = [
    Document(
        page_content="LangChain v0.3 was released with LCEL improvements.",
        metadata={"last_accessed_at": datetime.datetime(2025, 1, 1)}
    ),
    Document(
        page_content="LangChain v0.4 adds native streaming support.",
        metadata={"last_accessed_at": datetime.datetime(2025, 6, 1)}  # more recent
    ),
]
retriever.add_documents(docs)

# More recent documents get a score boost
results = retriever.invoke("Latest LangChain features")
# v0.4 doc ranks higher due to recency boost

9 — Combining retrievers in a full RAG chain


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── 1. Ingest ─────────────────────────────────────────────
loader = PyPDFLoader("reports/annual_report.pdf")
docs = loader.load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
).split_documents(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")

# ── 2. Build a strong retriever ───────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Layer 1: Multi-query to widen recall
multi_query = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)

# Layer 2: Ensemble with BM25 for keyword precision
bm25 = BM25Retriever.from_documents(chunks, k=4)
ensemble = EnsembleRetriever(
    retrievers=[bm25, multi_query],
    weights=[0.3, 0.7]
)

# Layer 3: Compression to remove noise
compression_retriever = ContextualCompressionRetriever(
    base_compressor=EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76),
    base_retriever=ensemble
)

# ── 3. RAG chain ──────────────────────────────────────────
def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source','?')}, p.{d.metadata.get('page','?')}]\n{d.page_content}"
        for d in docs
    )

prompt = ChatPromptTemplate.from_template("""
Answer using only the context provided.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
""")

chain = (
    {"context": compression_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("What was the total revenue in Q4 2024?"))

A Retriever in LangChain is responsible for fetching the most relevant documents from a vector store or other knowledge source based on a user query. It acts as the bridge between stored embeddings and the LLM, enabling efficient semantic retrieval for RAG applications. LangChain supports simple vector retrievers as well as advanced retrievers such as MultiQueryRetriever, ParentDocumentRetriever, ContextualCompressionRetriever, and SelfQueryRetriever.